# Multi-Condition Analysis with MultiNicheNet

## Biological Question

Here we ask: **which ligand-receptor programs change reproducibly between contralateral (C) and normal adjacent (N) tissue across biological samples?** Unlike Tutorial 02, this workflow treats sample identity as part of the experimental design and uses replicate-aware pseudobulk differential analysis.

This tutorial is appropriate when the scientific claim depends on variation across patients, samples, organoids, perturbation replicates, or batches that should not be treated as independent cells.

## Learning Objectives

By the end of this tutorial, you will be able to:

- Decide when to use MultiNicheNet instead of standard NicheNet
- Verify that your data has sufficient biological replicates
- Run replicate-aware differential cell-cell communication analysis
- Interpret the composite prioritization score
- Extract MultiNicheNet result tables from a `ContextResult`
- Prepare results for Sankey visualization with ccflowR

## Overview

We study **signaling to epithelial cell layers** (basal, parabasal, suprabasal, dividing) comparing contralateral (C) vs normal adjacent (N) tissue in the HNSCC dataset. The biological question is similar to Tutorial 02, but the statistical unit is now the sample rather than the cell.

> **When to use this workflow:**
> - You have multiple conditions and biological replicates.
> - You want statistically robust, replicate-aware differential communication.
> - You want to prioritize ligand-receptor pairs using expression, differential expression, ligand activity, and cell abundance together.
>
> **Use [02-standard-workflow](02-standard-workflow.ipynb) instead if:**
> - You have no biological replicates.
> - You want a simpler single receiver-cell analysis.

## When to Use NicheNet vs MultiNicheNet

| Feature | NicheNet | MultiNicheNet |
|---------|----------|---------------|
| Function | `predict_ligand_activity_nichenet()` | `predict_ligand_activity_multinichenet()` |
| Replicates | Not required | **Required** |
| Output | Ligand activity scores (AUPR) | Differential ligand prioritization |
| DE method | Wilcoxon rank-sum | Pseudobulk (EdgeR) |
| `sample_col` | Not needed | **Required** |
| Use case | Single condition or no replicates | Multi-condition with replicates |

## Setup

In [ ]:
suppressPackageStartupMessages({
    library(ContextNet)
    library(Seurat)
    library(multinichenetr)
    library(dplyr)
    library(ggplot2)
})

theme_set(theme_minimal(base_family = "sans"))


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                    USER CONFIGURATION                       ║
# ║  Paths are relative to the ContextNet project root.          ║
# ╚══════════════════════════════════════════════════════════════╝

PROJECT_DIR <- normalizePath(getwd(), mustWork = TRUE)
if (basename(PROJECT_DIR) == "notebooks" &&
    file.exists(file.path(PROJECT_DIR, "..", "DESCRIPTION"))) {
    PROJECT_DIR <- normalizePath(file.path(PROJECT_DIR, ".."), mustWork = TRUE)
}
if (!file.exists(file.path(PROJECT_DIR, "DESCRIPTION"))) {
    stop("Run this notebook from the ContextNet repository root or from the notebooks/ directory.")
}
setwd(PROJECT_DIR)
cat("Working directory:", getwd(), "
")

DATA_PATH       <- "data"
GRN_FILE        <- file.path(DATA_PATH, "gene_x_tf.csv")
SEURAT_FILE     <- file.path(DATA_PATH, "seurat_obj.rds")
NICHENET_DB     <- DATA_PATH

# Analysis parameters: replicate-aware multi-condition comparison
SENDER          <- NULL           # Auto-detect: all cell types with >500 cells
RECEIVER        <- c("basal", "parabasal", "suprabasal", "dividing")
GROUP_COL       <- "Fine_Cell_Types"
IDENT_COL       <- "category"     # Condition column
IDENT_1         <- "C"            # Contralateral (condition of interest)
IDENT_2         <- "N"            # Normal adjacent (reference)
SAMPLE_COL      <- "sample"       # REQUIRED for pseudobulk DE
MIN_CELLS       <- 500            # Minimum cells for sender cell types
TOP_N_LIGANDS   <- 30
N_CORES         <- 20             # Parallel cores for multinichenetr

stopifnot(file.exists(GRN_FILE), file.exists(SEURAT_FILE))


## Step 1: Load and Explore Data

MultiNicheNet requires biological replicates. Verify that each condition has multiple samples.

In [3]:
seurat_obj <- readRDS(SEURAT_FILE)
seurat_obj

An object of class Seurat 
16524 features across 145769 samples within 1 assay 
Active assay: RNA (16524 features, 0 variable features)
 2 layers present: counts, data
 2 dimensional reductions calculated: pca, umap

In [4]:
# Verify replicates: each condition must have multiple samples
table(seurat_obj$category, seurat_obj[[SAMPLE_COL]][, 1])

   
    Pt32C Pt32M Pt33C Pt33M Pt44C Pt44N Pt44M Pt45C Pt45N Pt45T Pt50C Pt50N
  C  6182     0  2498     0  6444     0     0  6534     0     0   942     0
  M     0  9236     0  3959     0     0  6860     0     0     0     0     0
  N     0     0     0     0     0 14515     0     0  4210     0     0  2239
  T     0     0     0     0     0     0     0     0     0 16104     0     0
   
    Pt50T Pt51C Pt51N Pt51M Pt56C Pt56N Pt56T Pt59C Pt59N Pt59T
  C     0  5439     0     0  6071     0     0  4716     0     0
  M     0     0     0  6596     0     0     0     0     0     0
  N     0     0  5580     0     0  1055     0     0  8175     0
  T  7463     0     0     0     0     0  8201     0     0 12750

### Replicate Design Check

MultiNicheNet treats samples, not cells, as the experimental units. The table above should show multiple samples per condition; otherwise the workflow cannot support replicate-aware differential communication claims.


In [5]:
# Subset to conditions of interest
seurat_cn <- subset(seurat_obj, subset = category %in% c(IDENT_1, IDENT_2))
seurat_cn[[SAMPLE_COL]] <- droplevels(as.factor(seurat_cn[[SAMPLE_COL]][, 1]))
seurat_cn$category <- droplevels(as.factor(seurat_cn$category))

cat("Cells:", ncol(seurat_cn), "\n")
table(seurat_cn$category)

Cells: 74600 



    C     N 
38826 35774 

In [6]:
celltype_counts <- sort(table(seurat_cn[[GROUP_COL]][, 1]), decreasing = TRUE)
celltype_counts


        basal   Endothelial        T Cell    suprabasal Myofibroblast 
        16039         13538         13031         12471          5103 
    parabasal    Fibroblast      dividing       Myeloid transitioning 
         3282          3109          2971          2698          1816 
   Melanocyte        B Cell        Muscle      Neuronal 
          234           151           120            37 

### Sender and Receiver Universe

The analysis tests communication from abundant sender populations to epithelial receiver states. The sender list controls the source populations considered; the receiver vector controls where condition-specific target programs are evaluated.


In [ ]:
if (is.null(SENDER)) {
    SENDER <- names(celltype_counts)[celltype_counts > MIN_CELLS]
}

cat("Senders:", paste(SENDER, collapse = ", "), "
")
cat("Receivers:", paste(RECEIVER, collapse = ", "), "
")
cat("Comparison:", IDENT_1, "vs", IDENT_2, "
")


## Step 2: Import GRN and Integrate

Same as the standard workflow — the context-specific ligand-target matrix is shared across NicheNet and MultiNicheNet. If you ran Tutorial 02 first, the saved network is loaded to skip the ~10 min integration step.

> **Bring your own workflow:** You can extract the ligand-target matrix with `ligand_target_matrix(network)` and use it directly in a [multinichenetr workflow](https://github.com/saeyslab/multinichenetr/blob/main/vignettes/basic_analysis_steps_MISC.md). ContextNet's wrapper (Step 3) handles gene name sanitization, bidirectional contrasts, and LR network loading automatically, but is optional.

In [7]:
# Reuse network from Tutorial 02 if available (saves ~10 min)
if (file.exists("/tmp/contextnet_network.rds")) {
    network <- readRDS("/tmp/contextnet_network.rds")
    cat("Loaded pre-built network\n")
} else {
    grn <- import_grn(GRN_FILE)
    network <- integrate_networks(grn, nichenet_database = NICHENET_DB, mode = "replace")
}
network

→ Loading GRN from: /fh/fast/setty_m/user/yhuang2/ContextNet/data/gene_x_tf.csv

ℹ Matrix dimensions: 17226 genes x 701 TFs

→ Normalizing weights per TF...

→ Converting to edge list...

✔ Imported GRN:

TFs: 701

Targets: 11607

Edges: "2,334,753"



── Network Integration ──



→ Loading NicheNet networks...

ℹ Loading from: /fh/fast/setty_m/user/yhuang2/ContextNet/data

✔ Loaded lr_network from lr_network_human_21122021.rds

✔ Loaded weighted_networks from weighted_networks_nsga2r_final.rds

ℹ Ligands from lr_network: 1226

ℹ GRN edges: "2,334,753"

ℹ GRN TFs: 701

ℹ GRN targets: 11607

→ Integrating networks (mode: "replace")...

ℹ PPR damping_factor: 0.7886

ℹ LTF cutoff: 0.9256

→ Constructing ligand-target matrix (this may take a while)...

✔ Constructed ligand-target matrix:

Ligands: 1226

Targets: 25337

Non-zero entries: "14,217,826"



── ContextNetwork ──────────────────────────────────────────────────────────────



Ligands: "1,226"

Targets: "25,337"

Integration mode: 

## Step 3: Run MultiNicheNet

Key differences from NicheNet:
- `sample_col` is **required** for pseudobulk differential expression
- `receiver` can be a vector — MultiNicheNet analyzes all sender-receiver pairs simultaneously
- ContextNet automatically handles cell type name sanitization (spaces to dots), gene alias conversion, and bidirectional contrast computation

In [ ]:
# ~20-40 minutes depending on dataset size and n_cores
result <- predict_ligand_activity_multinichenet(
    object = seurat_cn,
    network = network,
    sender = SENDER,
    receiver = RECEIVER,
    group_col = GROUP_COL,
    ident_col = IDENT_COL,
    ident_1 = IDENT_1,
    ident_2 = IDENT_2,
    sample_col = SAMPLE_COL,          # REQUIRED: identifies biological replicates
    nichenet_database = NICHENET_DB,
    top_n_ligands = TOP_N_LIGANDS,
    n_cores = N_CORES
)


## Step 4: Explore Results

MultiNicheNet returns a prioritization table with composite scores integrating expression, differential expression, ligand activity, and cell abundance. Results span all sender-receiver pairs and both contrast directions (C-N and N-C).

`result` is still a `ContextResult`, but its primary table comes from MultiNicheNet prioritization rather than a simple NicheNet ligand-activity ranking. The object records the sender/receiver universe, condition contrast, sample column, and thresholds used to generate the prioritization.

Use the same accessors where possible:

| Accessor | What it returns here | Interpretation |
|----------|----------------------|----------------|
| `ligand_activities(result)` | activity/prioritization-associated ligand table | differential candidate signals |
| `top_ligands(result)` | prioritized ligands | ligands to carry into plots and validation |
| `ligand_target_links(result)` | ligand-target evidence where available | mechanistic target support |
| `parameters(result)` | sample-aware analysis settings | exact replicate-aware contrast |
| `as.data.frame(result)` | exportable result table | generic table for filtering/reporting |

For MultiNicheNet, always interpret results at the sender-receiver-condition level. A high-scoring ligand can reflect a combination of ligand expression, receptor expression, ligand activity, and condition-specific changes; it is not equivalent to a ligand-only DE result.

In [ ]:
# Extract exportable data from the MultiNicheNet ContextResult
multi_tbl <- as.data.frame(result)
activity_tbl <- ligand_activities(result)
links_tbl <- ligand_target_links(result)
analysis_params <- parameters(result)

cat("Result rows:", nrow(multi_tbl), "\n")
cat("Activity rows:", nrow(activity_tbl), "\n")
cat("Ligand-target links:", nrow(links_tbl), "\n")

head(multi_tbl, 5)

In [ ]:
activities <- ligand_activities(result)

ligand_col <- if ("ligand" %in% colnames(activities)) "ligand" else "test_ligand"
activity_col <- if ("activity_scaled" %in% colnames(activities)) "activity_scaled" else "aupr_corrected"
contrast_col <- intersect(c("contrast", "group", "direction_regulation"), colnames(activities))

ligand_summary <- activities %>%
    group_by(.data[[ligand_col]]) %>%
    summarize(
        activity_score = max(.data[[activity_col]], na.rm = TRUE),
        n_rows = dplyr::n(),
        .groups = "drop"
    ) %>%
    arrange(desc(activity_score))

colnames(ligand_summary)[1] <- "ligand"
head(ligand_summary, 15)


### Reading the MultiNicheNet Table

For this workflow, a high-ranked row should be read as a **sample-aware differential communication program**. The score is not only ligand activity; it combines ligand/receptor expression, differential regulation, target activity, and sender-receiver context. Use the table to identify which component supports the path before making a biological claim.


In [ ]:
top_ligands(result)

In [ ]:
head(top_targets(result), 20)

In [ ]:
parameters(result)

## Step 5: Visualize

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 8)

top_act <- head(ligand_summary, 30)
top_act$ligand <- factor(top_act$ligand, levels = rev(top_act$ligand))

ggplot(top_act, aes(x = activity_score, y = ligand)) +
    geom_col(fill = "#2C7BB6") +
    labs(x = "Ligand Activity / Prioritization Score", y = NULL,
         title = "Top 30 Differential Ligands (Epithelial Receivers, C vs N)") +
    theme_minimal() +
    theme(plot.title = element_text(hjust = 0.5, face = "bold"))


**Interpretation**: Unlike NicheNet's raw AUPR, MultiNicheNet's `activity_scaled` is normalized across all tested ligands and conditions. Higher values indicate ligands whose predicted targets are more differentially expressed in the contralateral vs normal adjacent comparison. The composite prioritization score (shown in the parameter table below) further integrates expression, DE, and ligand activity into a single ranking.

### Sankey Visualization with ccflowR

ccflowR is especially useful for MultiNicheNet because the result table spans many sender-receiver pairs. The graph asks: **which sender -> ligand -> receptor -> epithelial receiver routes account for the highest-priority differential communication programs across samples?**

`prepare_for_ccflowr(framework = "multinichenet")` maps the MultiNicheNet prioritization table into path rows. `combine_sankey_data()` then rescales and thresholds those rows for visualization.

Interpretation rules:

- Compare flows only within the same thresholding scheme.
- A wide flow highlights a prioritized differential program, not direct evidence of physical contact.
- The most useful follow-up is to return to `multi_tbl` and inspect ligand/receptor expression, differential scores, and target evidence for the visible paths.


In [ ]:
sankey_raw <- prepare_for_ccflowr(
    result,
    expression_data = seurat_cn,
    network = network,
    group_col = GROUP_COL,
    top_n_ligands = 50,
    framework = "multinichenet"
)

sankey_data <- combine_sankey_data(sankey_raw, threshold_quantile = 0.95)

cat("Raw ccflowR path rows:", nrow(sankey_raw), "
")
cat("Rows after thresholding:", nrow(sankey_data), "
")
head(sankey_data)


In [ ]:
if (requireNamespace("ccflowR", quietly = TRUE)) {
    cell_types_in_plot <- unique(c(sankey_data$sender, sankey_data$receiver))
    palette <- grDevices::hcl.colors(length(cell_types_in_plot), palette = "Dark 3")
    cell_colors <- stats::setNames(palette, cell_types_in_plot)
    ccflowR::sankey_flow(sankey_data, cell_colors)
} else {
    message("ccflowR is not installed. Install it to render the Sankey graph: remotes::install_github('settylab/ccflowR')")
}


## Step 6: Save Results

In [ ]:
tmp_dir <- tempdir()
save_contextnet(result, file.path(tmp_dir, "multinichenet_result.cnet"))
write.csv(ligand_activities(result), file.path(tmp_dir, "multinichenet_activities.csv"), row.names = FALSE)
cat("Saved to:", tmp_dir, "\n")

In [ ]:
sessionInfo()

## Key Parameters

| Parameter | Required | Description |
|-----------|----------|-------------|
| `sender` | Yes | Sender cell type(s) expressing ligands |
| `receiver` | Yes | Receiver cell type |
| `ident_col` | Yes | Metadata column for condition grouping |
| `ident_1` | Yes | Condition of interest |
| `ident_2` | Yes | Reference condition |
| `sample_col` | **Yes** | Metadata column for sample/replicate IDs |
| `group_col` | No | Metadata column with cell type labels (default: `"cell_type"`) |
| `top_n_ligands` | No | Number of top ligands (default: 30) |
| `empirical_pval` | No | Use permutation-based p-values (default: FALSE) |
| `min_cells` | No | Minimum cells per cell type per sample (default: 10) |
| `nichenet_database` | No | Path to NicheNet data files |

### MultiNicheNet Prioritization Score

The composite prioritization score integrates criteria such as ligand expression, receptor expression, ligand differential expression, receptor differential expression, ligand activity, and cell abundance. Inspect the underlying table before assigning the biological signal to one mechanism.

| Criterion | Description |
|-----------|-------------|
| `scaled_lfc_ligand` | Scaled log fold-change of ligand between conditions |
| `scaled_p_val_ligand_adapted` | Scaled significance of ligand DE |
| `scaled_lfc_receptor` | Scaled log fold-change of receptor between conditions |
| `scaled_p_val_receptor_adapted` | Scaled significance of receptor DE |
| `max_scaled_activity` | Maximum ligand activity score across target genes |
| `scaled_pb_ligand` | Scaled pseudobulk expression of ligand |
| `scaled_pb_receptor` | Scaled pseudobulk expression of receptor |
| `fraction_expressing_ligand_receptor` | Fraction of cells expressing the L-R pair |

## Using the Context-Specific Matrix with multinichenetr Directly

If you need features beyond ContextNet's wrapper, extract the context-specific ligand-target matrix and use it in multinichenetr directly:

```r
ltm <- ligand_target_matrix(network)

# Use in multinichenetr's multi_nichenet_analysis()
# by passing ltm as the ligand_target_matrix argument.
```

This is useful for custom contrasts, paired designs, additional filtering, or direct comparison between the ContextNet matrix and the default NicheNet matrix.

For complete examples, see the multinichenetr tutorials:
- [Basic analysis](https://github.com/saeyslab/multinichenetr/blob/main/vignettes/basic_analysis_steps_MISC.md) - full walkthrough with MISC dataset
- [Pairwise comparison](https://github.com/saeyslab/multinichenetr/blob/main/vignettes/pairwise_analysis_MISC.md) - condition-pair analysis
- [Paired design](https://github.com/saeyslab/multinichenetr/blob/main/vignettes/paired_analysis_SCC.knit.md) - paired samples, such as tumor vs normal from the same patient